In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

import corner
from pydantic_yaml import parse_yaml_raw_as

from sklearn.neighbors import KDTree

import pyccl as ccl

from c2i2o.functions.scipy_wrap import ScipyWrapped, SCIPY_MODELS, SCIPY_UNION
from c2i2o.data_model import prior_set

In [ ]:
with open('prior_set.yaml', 'r') as fin:
    data = fin.read()
    ps = parse_yaml_raw_as(prior_set.PriorSet, data)

In [ ]:
t_dict = ps.generate_data(1000)
col_names, t_array = prior_set.convert_dict_to_2d_array(t_dict)
t_use = t_array[:,4:]


In [ ]:
col_names

In [ ]:
tree = KDTree(t_use)

In [ ]:
tree.query(t_use[3].reshape(1, -1), 3)

In [ ]:
_ = corner.corner(t_dict, var_names=col_names[4:])

In [ ]:
_cosmo_inputs = prior_set.convert_table_to_list_of_dicts(t_dict)

In [ ]:
a_vals = np.linspace(0.1, 1., 100)
k_vals = np.logspace(-4, 1, 101)
ref_cosmo = ccl.CosmologyVanillaLCDM()
chi_list = []
out_list = []
ref_power = ref_cosmo.nonlin_power(k_vals, a_vals).T
ref_chi = ref_cosmo.comoving_radial_distance(a_vals)
for i in range(50):
  if i == 0: 
    pass
  elif i%10 == 0:
    sys.stdout.write('.')
    sys.stdout.flush()
  cosmo = ccl.Cosmology(**_cosmo_inputs[i])
  data = cosmo.nonlin_power(k_vals, a_vals).T
  chi_list.append(cosmo.comoving_radial_distance(a_vals) / ref_chi)
  out_list.append(data/ref_power)
out_data = np.dstack(out_list)
chi_data = np.vstack(chi_list)

In [ ]:
_ = plt.imshow(out_data.std(axis=2)/out_data.mean(axis=2), aspect="auto", origin="lower", extent=(a_vals[0], a_vals[-1], k_vals[0], k_vals[-1]))
_ = plt.colorbar()
_ = plt.yscale('log')

In [ ]:
_ = plt.imshow(out_data[:,:,3], aspect="auto", origin="lower", extent=(a_vals[0], a_vals[-1], k_vals[0], k_vals[-1]))
_ = plt.colorbar()
_ = plt.yscale('log')

In [ ]:
_ = plt.scatter(chi_data.mean(axis=0), chi_data.std(axis=0))

In [ ]:
np.vstack(chi_list)